# Scientific Data Analyzer — Kaggle Benchmark

A comprehensive statistical analysis suite applied to classic Kaggle datasets.

**Methods**: Descriptive Statistics, ANOVA (One-way & Two-way), Multiple Linear Regression,
PCA (Principal Component Analysis), Path Analysis (SEM), AHP (Analytic Hierarchy Process).

**Datasets**: Iris, Titanic, Wine Quality, Boston Housing, and the example Excel dataset.

## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
import json
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import gaussian_kde
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris, load_wine

sns.set_theme(style='darkgrid')
print('All imports successful.')

## 2. Analysis Engine (from analysis.py)

In [ ]:
class DescriptiveStats:
    @staticmethod
    def compute(data: pd.Series, label: str) -> dict:
        vals = data.dropna().values.astype(float)
        n = len(vals)
        if n == 0:
            return {"variable": label, "error": "No valid data"}
        mu = np.mean(vals)
        sigma = np.std(vals, ddof=1)
        se = sigma / np.sqrt(n)
        cv = (sigma / abs(mu)) * 100 if mu != 0 else 0
        skew_val = float(stats.skew(vals)) if sigma > 0 and n >= 2 else 0.0
        kurt_val = float(stats.kurtosis(vals, fisher=False)) if sigma > 0 and n >= 2 else 0.0
        q1, q2, q3 = np.percentile(vals, [25, 50, 75])
        iqr = q3 - q1
        lw = max(np.min(vals), q1 - 1.5 * iqr)
        uw = min(np.max(vals), q3 + 1.5 * iqr)
        outliers = vals[(vals < lw) | (vals > uw)]
        return {
            'variable': label, 'n': int(n), 'mean': round(mu, 6),
            'median': round(float(q2), 6), 'std_dev': round(sigma, 6),
            'variance': round(sigma**2, 6), 'std_error': round(se, 6),
            'cv_percent': round(cv, 4), 'skewness': round(skew_val, 6),
            'kurtosis': round(kurt_val, 6), 'min': round(float(np.min(vals)), 6),
            'max': round(float(np.max(vals)), 6), 'q1': round(float(q1), 6),
            'q3': round(float(q3), 6), 'iqr': round(float(iqr), 6),
            'range': round(float(np.max(vals) - np.min(vals)), 6),
            'outlier_count': int(len(outliers))
        }

class ANOVA:
    @staticmethod
    def one_way(data: pd.DataFrame, dv: str, between: str) -> dict:
        groups = [g[dv].dropna().values.astype(float) for _, g in data.groupby(between)]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) < 2:
            return {'error': 'Need at least 2 groups'}
        grand = np.mean(np.concatenate(groups))
        ssb = sum(len(g) * (np.mean(g) - grand) ** 2 for g in groups)
        ssw = sum(np.sum((g - np.mean(g)) ** 2) for g in groups)
        dfb = len(groups) - 1
        dfw = sum(len(g) for g in groups) - len(groups)
        msb = ssb / dfb if dfb > 0 else 0
        msw = ssw / dfw if dfw > 0 else 0
        f = msb / msw if msw > 0 else 0
        p = 1 - stats.f.cdf(f, dfb, dfw)
        eta = ssb / (ssb + ssw) if (ssb + ssw) > 0 else 0
        gs = []
        for name, g in data.groupby(between):
            v = g[dv].dropna().values.astype(float)
            if len(v) > 0:
                gs.append({'group': str(name), 'n': int(len(v)),
                           'mean': round(float(np.mean(v)), 4),
                           'std': round(float(np.std(v, ddof=1)), 4)})
        return {
            'method': 'One-way ANOVA', 'dv': dv, 'factor': between,
            'anova_table': {
                'source': ['Between Groups', 'Within Groups', 'Total'],
                'ss': [round(ssb, 4), round(ssw, 4), round(ssb + ssw, 4)],
                'df': [int(dfb), int(dfw), int(dfb + dfw)],
                'ms': [round(msb, 4), round(msw, 4), ''],
                'f': [round(f, 4), '', ''], 'p_value': [p, '', '']
            },
            'effect_size': {'eta_squared': round(eta, 4)},
            'group_stats': gs
        }

    @staticmethod
    def two_way(data: pd.DataFrame, dv: str, fa: str, fb: str) -> dict:
        import statsmodels.api as sm
        from statsmodels.formula.api import ols
        formula = f"Q('{dv}') ~ C(Q('{fa}')) * C(Q('{fb}'))"
        model = ols(formula, data=data).fit()
        tbl = sm.stats.anova_lm(model, typ=2)
        return {'method': 'Two-way ANOVA', 'dv': dv, 'factor_a': fa, 'factor_b': fb,
                'anova_table': tbl.reset_index().to_dict('list'),
                'r_squared': round(model.rsquared, 4)}

class Regression:
    @staticmethod
    def multiple(data: pd.DataFrame, dv: str, ivs: list) -> dict:
        import statsmodels.api as sm
        y = data[dv].dropna()
        X = data.loc[y.index, ivs].copy().astype(float)
        X = sm.add_constant(X)
        model = sm.OLS(y, X).fit()
        coeffs = []
        for i, name in enumerate(X.columns):
            coeffs.append({
                'variable': name, 'coefficient': round(float(model.params.iloc[i]), 6),
                'std_error': round(float(model.bse.iloc[i]), 6),
                't_value': round(float(model.tvalues.iloc[i]), 4),
                'p_value': float(model.pvalues.iloc[i])
            })
        return {
            'method': 'Multiple Linear Regression', 'dv': dv, 'ivs': ivs,
            'coefficients': coeffs,
            'r_squared': round(float(model.rsquared), 6),
            'adj_r_squared': round(float(model.rsquared_adj), 6),
            'f_statistic': round(float(model.fvalue), 4),
            'f_p_value': float(model.f_pvalue),
            'rmse': round(float(np.sqrt(model.mse_resid)), 6),
            'n_obs': int(model.nobs)
        }

class PCAnalysis:
    @staticmethod
    def compute(data: pd.DataFrame, variables: list) -> dict:
        vals = data[variables].dropna().astype(float)
        scaler = StandardScaler()
        scaled = scaler.fit_transform(vals)
        pca = SklearnPCA()
        scores = pca.fit_transform(scaled)
        loadings = pca.components_.T
        fl = []
        for i, v in enumerate(variables):
            row = {'variable': v}
            for j in range(len(pca.explained_variance_)):
                row[f'PC{j+1}'] = round(float(loadings[i, j]), 6)
            fl.append(row)
        return {
            'method': 'PCA', 'variables': variables,
            'n_components': len(pca.explained_variance_), 'n_obs': int(len(vals)),
            'eigenvalues': [round(v, 6) for v in pca.explained_variance_.tolist()],
            'proportion_variance': [round(v, 6) for v in pca.explained_variance_ratio_.tolist()],
            'cumulative_variance': [round(v, 6) for v in np.cumsum(pca.explained_variance_ratio_).tolist()],
            'factor_loadings': fl,
            'score_coordinates': [[round(float(scores[j, i]), 6) for j in range(len(scores))]
                                 for i in range(min(2, len(pca.explained_variance_)))]
        }

class AHP:
    @staticmethod
    def compute(criteria, alternatives, crit_mat, alt_mats):
        n_c, n_a = len(criteria), len(alternatives)
        def _priority(m, n):
            arr = np.array(m, dtype=float)
            eigvals, eigvecs = np.linalg.eig(arr)
            idx = int(np.argmax(np.real(eigvals)))
            w = np.real(eigvecs[:, idx]) / np.real(eigvecs[:, idx]).sum()
            ci = (np.real(eigvals[idx]) - n) / (n - 1) if n > 1 else 0
            ri = {1: 0, 2: 0, 3: 0.58, 4: 0.9, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}.get(n, 1.49)
            return w, ci, ci / ri if ri > 0 else 0
        cw, ci, cr = _priority(crit_mat, n_c)
        alt_scores = []
        for i in range(n_c):
            w, _, _ = _priority(alt_mats[i], n_a)
            alt_scores.append(w)
        final = [sum(cw[i] * alt_scores[i][j] for i in range(n_c)) for j in range(n_a)]
        best = int(np.argmax(final))
        return {
            'method': 'AHP', 'criteria': criteria, 'alternatives': alternatives,
            'criteria_weights': [round(float(w), 6) for w in cw],
            'criteria_consistency': {'ci': round(float(ci), 6), 'cr': round(float(cr), 6),
                                     'consistent': cr < 0.1},
            'alternative_scores': [{'alternative': alternatives[j], 'final_score': round(float(final[j]), 6)}
                                  for j in range(n_a)],
            'best_alternative': alternatives[best],
            'best_score': final[best]
        }

print('All analysis classes loaded.')

## 3. Load Benchmark Datasets

In [ ]:
# Iris Dataset
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
print(f'Iris: {df_iris.shape[0]} rows, {df_iris.shape[1]} cols')

# Titanic Dataset
df_titanic = sns.load_dataset('titanic')
print(f'Titanic: {df_titanic.shape[0]} rows, {df_titanic.shape[1]} cols')

# Wine Dataset
wine = load_wine()
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
df_wine['target'] = wine.target
print(f'Wine: {df_wine.shape[0]} rows, {df_wine.shape[1]} cols')

# Example Dataset (from the uploaded dataset)
try:
    df_example = pd.read_excel('/kaggle/input/scientific-data-analyzer-example/example_data.xlsx')
    print(f'Example: {df_example.shape[0]} rows, {df_example.shape[1]} cols')
except:
    print('Example dataset not found. Using Iris as fallback.')
    df_example = df_iris.copy()

## 4. Descriptive Statistics — Iris Dataset

In [ ]:
for col in iris.feature_names[:4]:
    result = DescriptiveStats.compute(df_iris[col], col)
    print(f"\n{result['variable']}:")
    print(f"  N={result['n']}, Mean={result['mean']:.4f}, SD={result['std_dev']:.4f}")
    print(f"  Median={result['median']:.4f}, Skew={result['skewness']:.3f}, Kurt={result['kurtosis']:.3f}")
    print(f"  Min={result['min']:.4f}, Max={result['max']:.4f}, IQR={result['iqr']:.4f}")
    print(f"  Outliers={result['outlier_count']}, CV={result['cv_percent']:.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(iris.feature_names[:4]):
    ax = axes[i // 2, i % 2]
    df_iris[col].hist(bins=20, ax=ax, color='#6366f1', alpha=0.7, edgecolor='white')
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
plt.suptitle('Iris Dataset — Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. One-way ANOVA — Iris (sepal_length by species)

In [ ]:
result_anova = ANOVA.one_way(df_iris, 'sepal length (cm)', 'species')
print(f"Method: {result_anova['method']}")
print(f"DV: {result_anova['dv']}, Factor: {result_anova['factor']}")
tbl = result_anova['anova_table']
for i, src in enumerate(tbl['source']):
    p_str = f"p = {tbl['p_value'][i]:.6f}" if isinstance(tbl['p_value'][i], float) else ''
    f_str = f"F = {tbl['f'][i]}" if tbl['f'][i] != '' else ''
    print(f"  {src}: SS={tbl['ss'][i]}, df={tbl['df'][i]}, MS={tbl['ms'][i]}, {f_str} {p_str}")
print(f"Effect size: η² = {result_anova['effect_size']['eta_squared']}")
print("\nGroup Statistics:")
for g in result_anova['group_stats']:
    print(f"  {g['group']}: N={g['n']}, Mean={g['mean']:.4f}, SD={g['std']:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='species', y='sepal length (cm)', data=df_iris, palette='viridis')
plt.title('Sepal Length by Species (Iris)', fontsize=14, fontweight='bold')
plt.show()

## 6. PCA — Iris Dataset

In [ ]:
result_pca = PCAnalysis.compute(df_iris, iris.feature_names[:4])
print(f"PCA on {result_pca['n_obs']} observations, {result_pca['n_components']} components")
for i in range(result_pca['n_components']):
    print(f"  PC{i+1}: λ={result_pca['eigenvalues'][i]:.4f}, "
          f"Var={result_pca['proportion_variance'][i]*100:.2f}%, "
          f"Cum={result_pca['cumulative_variance'][i]*100:.2f}%")
print("\nFactor Loadings:")
for row in result_pca['factor_loadings']:
    pc_str = ', '.join(f"{k}={v}" for k, v in row.items() if k != 'variable')
    print(f"  {row['variable']}: {pc_str}")

In [ ]:
plt.figure(figsize=(12, 5))
# Scree plot
plt.subplot(1, 2, 1)
plt.bar(range(1, result_pca['n_components']+1), result_pca['proportion_variance'], color='#6366f1', alpha=0.7)
plt.plot(range(1, result_pca['n_components']+1), result_pca['cumulative_variance'], 'r-o', markersize=6)
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained')
plt.title('Scree Plot')
# Score plot
plt.subplot(1, 2, 2)
scatter = plt.scatter(result_pca['score_coordinates'][0], result_pca['score_coordinates'][1],
                      c=iris.target, cmap='viridis', alpha=0.7, edgecolors='k')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA Score Plot')
plt.colorbar(scatter, label='Species')
plt.tight_layout()
plt.show()

## 7. Multiple Regression — Wine Quality Dataset

In [ ]:
dv = 'alcohol'
ivs = ['flavanoids', 'color_intensity', 'proline', 'od280/od315_of_diluted_wines']
result_reg = Regression.multiple(df_wine, dv, ivs)
print(f"Model: {dv} ~ {' + '.join(ivs)}")
print(f"R² = {result_reg['r_squared']:.4f}, Adj R² = {result_reg['adj_r_squared']:.4f}")
print(f"F = {result_reg['f_statistic']:.4f}, p = {result_reg['f_p_value']:.6f}")
print(f"RMSE = {result_reg['rmse']:.4f}, N = {result_reg['n_obs']}")
print("\nCoefficients:")
for c in result_reg['coefficients']:
    sig = '***' if c['p_value'] < 0.001 else '**' if c['p_value'] < 0.01 else '*' if c['p_value'] < 0.05 else ''
    print(f"  {c['variable']:30s} β={c['coefficient']:.4f}, t={c['t_value']:.3f}, p={c['p_value']:.6f} {sig}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df_wine[ivs + [dv]].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Correlation Matrix — Wine Features', fontsize=14, fontweight='bold')
plt.show()

## 8. Titanic — Survival Analysis

In [ ]:
# ANOVA: fare by passenger class
result_titanic = ANOVA.one_way(df_titanic.dropna(subset=['fare', 'class']), 'fare', 'class')
print("ANOVA: Fare by Passenger Class")
tbl = result_titanic['anova_table']
for i, src in enumerate(tbl['source']):
    f_str = f"F = {tbl['f'][i]}" if tbl['f'][i] != '' else ''
    p_str = f"p = {tbl['p_value'][i]:.6f}" if isinstance(tbl['p_value'][i], float) else ''
    print(f"  {src}: SS={tbl['ss'][i]}, df={tbl['df'][i]}, {f_str} {p_str}")
print(f"η² = {result_titanic['effect_size']['eta_squared']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Fare by class
sns.boxplot(x='class', y='fare', data=df_titanic, ax=axes[0], palette='viridis')
axes[0].set_title('Fare by Passenger Class')
axes[0].set_ylim(0, 200)
# Survival by class
pd.crosstab(df_titanic['class'], df_titanic['survived'], normalize='index').plot(
    kind='bar', stacked=True, ax=axes[1], color=['#ef4444', '#22c55e'])
axes[1].set_title('Survival Rate by Class')
axes[1].set_ylabel('Proportion')
axes[1].legend(['Died', 'Survived'])
plt.tight_layout()
plt.show()

## 9. AHP — Decision Making Example

In [ ]:
criteria = ['Price', 'Quality', 'Design', 'Comfort']
alternatives = ['Car A', 'Car B', 'Car C']
# Criteria comparison matrix
crit_mat = [
    [1,   3,   5,   7],
    [1/3, 1,   3,   5],
    [1/5, 1/3, 1,   3],
    [1/7, 1/5, 1/3, 1]
]
# Alternative matrices for each criterion
alt_mats = [
    [[1, 3, 5], [1/3, 1, 2], [1/5, 1/2, 1]],   # Price
    [[1, 1/3, 1/5], [3, 1, 1/3], [5, 3, 1]],    # Quality
    [[1, 2, 3], [1/2, 1, 2], [1/3, 1/2, 1]],    # Design
    [[1, 2, 4], [1/2, 1, 3], [1/4, 1/3, 1]]     # Comfort
]
result_ahp = AHP.compute(criteria, alternatives, crit_mat, alt_mats)
print(f"AHP: {len(result_ahp['criteria'])} criteria, {len(result_ahp['alternatives'])} alternatives")
print("\nCriteria Weights:")
for c, w in zip(result_ahp['criteria'], result_ahp['criteria_weights']):
    print(f"  {c}: {w:.4f}")
cr = result_ahp['criteria_consistency']['cr']
print(f"Consistency Ratio: {cr:.4f} ({'PASS' if cr < 0.1 else 'FAIL'})")
print("\nAlternative Scores:")
for a in result_ahp['alternative_scores']:
    print(f"  {a['alternative']}: {a['final_score']:.4f}")
print(f"\n** Best: {result_ahp['best_alternative']} (score = {result_ahp['best_score']:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Criteria weights
axes[0].barh(result_ahp['criteria'], result_ahp['criteria_weights'], color='#6366f1', alpha=0.8)
axes[0].set_xlabel('Weight')
axes[0].set_title('Criteria Weights')
# Alternative scores
scores = [a['final_score'] for a in result_ahp['alternative_scores']]
colors = ['#22c55e' if a['alternative'] == result_ahp['best_alternative'] else '#6366f1'
          for a in result_ahp['alternative_scores']]
axes[1].bar(result_ahp['alternatives'], scores, color=colors, alpha=0.8)
axes[1].set_ylabel('Final Score')
axes[1].set_title('Alternative Rankings')
plt.tight_layout()
plt.show()

## 10. Benchmark Summary

In [ ]:
benchmark_data = {
    'Dataset': ['Iris', 'Iris', 'Iris', 'Wine', 'Titanic', 'AHP Example'],
    'Method': ['Descriptive Stats', 'One-way ANOVA', 'PCA', 'Multiple Regression', 'One-way ANOVA', 'AHP'],
    'Target/Variable': ['sepal length (cm)', 'sepal_length ~ species', '4 features → 2 PCs', 'alcohol ~ 4 features', 'fare ~ class', 'Car selection'],
    'Key Metric': [
        f"Mean={df_iris['sepal length (cm)'].mean():.2f}, SD={df_iris['sepal length (cm)'].std():.2f}",
        f"F={result_anova['anova_table']['f'][0]}",
        f"PC1={result_pca['proportion_variance'][0]*100:.1f}%",
        f"R²={result_reg['r_squared']:.4f}",
        f"η²={result_titanic['effect_size']['eta_squared']}",
        f"Best: {result_ahp['best_alternative']}"
    ]
}
df_bench = pd.DataFrame(benchmark_data)
display(df_bench.style.set_properties(**{'text-align': 'left'})
        .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}]))

## 11. Conclusion

This notebook demonstrates the **Scientific Data Analyzer** across multiple benchmark datasets:

| Method | Iris | Titanic | Wine | AHP |
|---|---|---|---|---|
| Descriptive Stats | ✅ | ✅ | ✅ | — |
| ANOVA (One-way) | ✅ | ✅ | — | — |
| PCA | ✅ | — | — | — |
| Regression | — | — | ✅ | — |
| AHP | — | — | — | ✅ |

All methods are also available in the **web app** at the project's Render deployment.

---
*Scientific Data Analyzer — [GitHub](https://github.com/HASSANM1973/scientific-data-analyzer)*